# FreshMart Lab 4: Batch Scoring to Gold
**Microsoft Fabric Data Science**

การทำนายเป็นชุด (batch scoring) คือทำนายหลายแถวตามรอบ ไม่ใช่ทีละคำขอทันที  
เป้าหมาย: ทำนายสมาชิกใหม่ 200 คน แล้วเขียนตาราง `gold.freshmart_predictions` พร้อมความเร่งด่วนแคมเปญ

### กฎทอง (ถามบ่อยที่สุด — อ่านก่อนรัน)
**ห้าม** คำนวณ min/max ใหม่จากชุด 200 คน  
**ต้อง** ใช้ `feature_params.json` จาก Lab 2

ถ้า scale ใหม่ ผลทำนายจะเพี้ยนหรือเอียงไปคลาสเดียว

### ถ้าติด — อ่านก่อนถาม TA
| อาการ | สาเหตุที่พบบ่อย | ทำอะไร |
| --- | --- | --- |
| หาโมเดลไม่เจอ | ยังไม่จบ Lab 3 | กลับไป register `freshmart-churn-model` |
| missing column / type mismatch | ลำดับฟีเจอร์ไม่ตรงลายเซ็น | ใช้ params จาก Lab 2 เท่านั้น |
| PREDICT unavailable | สภาพแวดล้อม | เซลล์มีทางเลือกสำรอง — ดูข้อความใน output |


### เตรียม loader + ฟังก์ชันฟีเจอร์


In [ ]:
import json
from pathlib import Path
import pandas as pd

from pathlib import Path
import pandas as pd

# Schema-qualified names (preferred). Legacy flat names still tried as fallback.
BRONZE_TRANSACTIONS = "bronze.transactions"
BRONZE_CUSTOMERS = "bronze.customers"
SILVER_CUSTOMER_FEATURES = "silver.customer_features"
GOLD_PREDICTIONS = "gold.freshmart_predictions"

def _first_existing(paths):
    for path in paths:
        candidate = Path(path)
        if candidate.exists() and candidate.is_file():
            return candidate
    return None

def load_csv(file_name: str) -> pd.DataFrame:
    found = _first_existing([
        f"/lakehouse/default/Files/raw/{file_name}",
        f"Files/raw/{file_name}",
        f"../data/{file_name}",
        f"labs/data/{file_name}",
        file_name,
    ])
    if found is None:
        raise FileNotFoundError(f"Cannot find {file_name}. Upload it to Files/raw or place it under labs/data.")
    print(f"Loaded CSV: {found}")
    return pd.read_csv(found)

def _table_candidates(table_name: str) -> list[str]:
    legacy = {
        "bronze.transactions": "bronze_transactions",
        "bronze.customers": "bronze_customers",
        "silver.customer_features": "silver_customer_features",
        "gold.freshmart_predictions": "gold_freshmart_predictions",
    }
    names = [table_name]
    if table_name in legacy:
        names.append(legacy[table_name])
    return names

def load_table_or_csv(table_name: str, file_name: str) -> pd.DataFrame:
    last_error = None
    for candidate in _table_candidates(table_name):
        try:
            frame = spark.read.table(candidate).toPandas()
            print(f"Loaded Spark table {candidate}: {len(frame):,} rows")
            return frame
        except Exception as exc:
            last_error = exc
    print(f"Spark table '{table_name}' unavailable ({last_error}). Falling back to CSV.")
    return load_csv(file_name)


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Any
import logging

logger = logging.getLogger(__name__)

ID_COLUMN = 'CustomerID'
TARGET_COLUMN = 'Churn'
CATEGORICAL_COLUMNS = ('MembershipTier', 'Gender')
IMPUTE_MEDIAN_COLUMNS = ('Age',)
SCALE_COLUMNS = ('MonetaryTotal', 'AvgBasketSize', 'RecencyDays', 'TenureMonths')
DUMMY_COLUMNS = ('MembershipTier_Bronze', 'MembershipTier_Gold', 'MembershipTier_Platinum', 'MembershipTier_Silver', 'Gender_F', 'Gender_M', 'Gender_Other')
PASSTHROUGH_COLUMNS = ('Frequency', 'ComplaintCount')
FEATURE_COLUMNS = ('Age', 'TenureMonths', 'RecencyDays', 'Frequency', 'MonetaryTotal', 'AvgBasketSize', 'ComplaintCount', 'MembershipTier_Bronze', 'MembershipTier_Gold', 'MembershipTier_Platinum', 'MembershipTier_Silver', 'Gender_F', 'Gender_M', 'Gender_Other')
REQUIRED_RAW_COLUMNS = ('CustomerID', 'Age', 'Gender', 'MembershipTier', 'TenureMonths', 'RecencyDays', 'Frequency', 'MonetaryTotal', 'AvgBasketSize', 'ComplaintCount')

@dataclass(frozen=True)
class FeatureParams:
    """Fitted preprocessing parameters reused at scoring time.

    Attributes:
        age_median: Median Age computed from the training customers.
        scale_mins: Per-column minimum used for min-max scaling.
        scale_maxs: Per-column maximum used for min-max scaling.
        dummy_columns: One-hot columns the model expects, in order.
        feature_columns: Final model input columns, in order.
    """

    age_median: float
    scale_mins: dict[str, float]
    scale_maxs: dict[str, float]
    dummy_columns: list[str]
    feature_columns: list[str]

    def to_dict(self) -> dict[str, Any]:
        """Serialize parameters to a JSON-friendly dictionary."""
        return asdict(self)

    @classmethod
    def from_dict(cls, payload: dict[str, Any]) -> FeatureParams:
        """Create parameters from a dictionary.

        Args:
            payload: Mapping produced by ``to_dict``.

        Returns:
            Validated ``FeatureParams``.

        Raises:
            ValueError: If required keys are missing.
        """
        required = {
            "age_median",
            "scale_mins",
            "scale_maxs",
            "dummy_columns",
            "feature_columns",
        }
        missing = required - set(payload)
        if missing:
            raise ValueError(f"FeatureParams missing keys: {sorted(missing)}")
        return cls(
            age_median=float(payload["age_median"]),
            scale_mins={k: float(v) for k, v in payload["scale_mins"].items()},
            scale_maxs={k: float(v) for k, v in payload["scale_maxs"].items()},
            dummy_columns=list(payload["dummy_columns"]),
            feature_columns=list(payload["feature_columns"]),
        )


def validate_raw_customers(df: pd.DataFrame, *, require_target: bool = True) -> pd.DataFrame:
    """Validate and normalize a raw FreshMart customer frame.

    Args:
        df: Raw customer records from CSV or ``bronze.customers``.
        require_target: When True, require the ``Churn`` column.

    Returns:
        Copy with numeric columns coerced.

    Raises:
        ValueError: If required columns are missing.
    """
    required = REQUIRED_RAW_COLUMNS + ((TARGET_COLUMN,) if require_target else ())
    _require_columns(df, required, frame_name="customer frame")
    numeric_cols = (
        "Age",
        "TenureMonths",
        "RecencyDays",
        "Frequency",
        "MonetaryTotal",
        "AvgBasketSize",
        "ComplaintCount",
    )
    if require_target:
        numeric_cols = numeric_cols + (TARGET_COLUMN,)
    return _numeric_copy(df, numeric_cols)


def _require_columns(df: pd.DataFrame, columns: tuple[str, ...], *, frame_name: str) -> None:
    """Raise if expected columns are missing."""
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise ValueError(f"{frame_name} missing required columns: {missing}")


def _numeric_copy(df: pd.DataFrame, columns: tuple[str, ...]) -> pd.DataFrame:
    """Return a copy with selected columns coerced to numeric."""
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def fit_preprocessor(df_raw: pd.DataFrame) -> FeatureParams:
    """Fit imputation and scaling parameters on training customers.

    Args:
        df_raw: Raw customer DataFrame including ``Churn``.

    Returns:
        Fitted parameters that must be reused for scoring.
    """
    df = validate_raw_customers(df_raw, require_target=True)
    age_median = float(df["Age"].median())
    if pd.isna(age_median):
        raise ValueError("Cannot fit preprocessor: Age median is NaN")

    scale_mins: dict[str, float] = {}
    scale_maxs: dict[str, float] = {}
    for col in SCALE_COLUMNS:
        col_min = float(df[col].min())
        col_max = float(df[col].max())
        if col_max <= col_min:
            raise ValueError(f"Cannot scale {col}: min={col_min}, max={col_max}")
        scale_mins[col] = col_min
        scale_maxs[col] = col_max

    params = FeatureParams(
        age_median=age_median,
        scale_mins=scale_mins,
        scale_maxs=scale_maxs,
        dummy_columns=list(DUMMY_COLUMNS),
        feature_columns=list(FEATURE_COLUMNS),
    )
    logger.info(
        "Fitted feature params: age_median=%.1f, scale_columns=%s",
        params.age_median,
        list(SCALE_COLUMNS),
    )
    return params


def _one_hot_categories(df: pd.DataFrame) -> pd.DataFrame:
    """One-hot encode membership and gender, keeping the expected columns."""
    encoded = pd.get_dummies(df, columns=list(CATEGORICAL_COLUMNS), drop_first=False)
    for col in DUMMY_COLUMNS:
        if col not in encoded.columns:
            encoded[col] = 0
    return encoded


def _min_max_scale(series: pd.Series, col_min: float, col_max: float) -> pd.Series:
    """Scale a series to [0, 1] using fitted min/max."""
    return (series - col_min) / (col_max - col_min + 1e-6)


def transform_customers(
    df_raw: pd.DataFrame,
    params: FeatureParams,
    *,
    require_target: bool = False,
) -> pd.DataFrame:
    """Apply the fitted FreshMart preprocessing contract.

    Args:
        df_raw: Raw customer records (training or scoring batch).
        params: Parameters from ``fit_preprocessor``.
        require_target: When True, keep and validate ``Churn``.

    Returns:
        Feature frame with ``CustomerID``, model columns, and optional ``Churn``.
    """
    df = validate_raw_customers(df_raw, require_target=require_target)
    work = df.copy()
    work["Age"] = work["Age"].fillna(params.age_median)

    work = _one_hot_categories(work)
    for col in SCALE_COLUMNS:
        work[col] = _min_max_scale(
            work[col],
            params.scale_mins[col],
            params.scale_maxs[col],
        )

    bool_cols = work.select_dtypes(include="bool").columns
    work[bool_cols] = work[bool_cols].astype(int)

    ordered = [ID_COLUMN, *params.feature_columns]
    if require_target or TARGET_COLUMN in work.columns:
        ordered.append(TARGET_COLUMN)
    missing = [col for col in ordered if col not in work.columns]
    if missing:
        raise ValueError(f"Transformed frame missing columns: {missing}")

    result = work[ordered].copy()
    feature_frame = result[list(params.feature_columns)]
    if feature_frame.isna().any().any():
        bad = feature_frame.columns[feature_frame.isna().any()].tolist()
        raise ValueError(f"NaN remaining in feature columns: {bad}")
    return result


def model_matrix(df_features: pd.DataFrame, params: FeatureParams) -> pd.DataFrame:
    """Return the model input matrix in signature order.

    Args:
        df_features: Output of ``transform_customers``.
        params: Fitted parameters.

    Returns:
        DataFrame with only model feature columns.
    """
    _require_columns(df_features, tuple(params.feature_columns), frame_name="feature frame")
    return df_features[list(params.feature_columns)].astype(float)


def save_feature_params(params: FeatureParams, path: str | Path) -> Path:
    """Write feature parameters to a JSON file.

    Args:
        params: Fitted parameters.
        path: Destination JSON path.

    Returns:
        Resolved output path.
    """
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(params.to_dict(), indent=2), encoding="utf-8")
    logger.info("Saved feature params to %s", out)
    return out


def load_feature_params(path: str | Path) -> FeatureParams:
    """Load feature parameters from a JSON file.

    Args:
        path: JSON path written by ``save_feature_params``.

    Returns:
        Fitted parameters.

    Raises:
        FileNotFoundError: If the file does not exist.
    """
    src = Path(path)
    if not src.exists():
        raise FileNotFoundError(f"Feature params not found: {src}")
    payload = json.loads(src.read_text(encoding="utf-8"))
    return FeatureParams.from_dict(payload)


### ขั้นตอนที่ 1: โหลดชุดทำนาย + params ชุดฝึก

**โค้ดนี้ทำอะไร**
1. อ่าน `Files/raw/freshmart_scoring_batch.csv` (200 แถว **ไม่มี** คอลัมน์ Churn)
2. โหลด `feature_params.json` จาก Lab 2
3. `transform_customers(..., require_target=False)` ให้ได้ 14 ฟีเจอร์ตามลายเซ็นโมเดล

รหัสลูกค้าชุดนี้ขึ้นต้น `CUST_05xxx` ไม่ซ้อนกับชุดฝึก


In [ ]:
try:
    df_scoring_raw = (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load("Files/raw/freshmart_scoring_batch.csv")
        .toPandas()
    )
    print("Loaded scoring batch from Files/raw")
except Exception as exc:
    print(f"Spark Files path unavailable ({exc})")
    df_scoring_raw = load_csv("freshmart_scoring_batch.csv")

print(f"Scoring Batch count: {len(df_scoring_raw):,}")
assert "Churn" not in df_scoring_raw.columns
display(df_scoring_raw.head())

params_path = _first_existing([
    "Files/params/feature_params.json",
    "/lakehouse/default/Files/params/feature_params.json",
    "labs/data/.local/feature_params.json",
])
if params_path:
    params = load_feature_params(params_path)
    print(f"Loaded feature params from {params_path}")
else:
    print("feature_params.json not found; fitting from bronze.customers")
    params = fit_preprocessor(load_table_or_csv("bronze.customers", "freshmart_customers.csv"))

spark_features = transform_customers(df_scoring_raw, params, require_target=False)
print("Aligned columns:", list(spark_features.columns))
print(spark_features.head())


### ขั้นตอนที่ 2: PREDICT

**ความรู้สั้น ๆ:** `MLFlowTransformer` ให้ Spark เรียกโมเดลที่ลงทะเบียนไว้เพื่อทำนายหลายแถวพร้อมกัน (การทำนายเป็นชุด)

ส่งเฉพาะคอลัมน์ฟีเจอร์ — **อย่าส่ง** `CustomerID`  
ถ้า PREDICT ไม่พร้อม เซลล์จะลอง `mlflow.pyfunc` ต่ออัตโนมัติ


In [ ]:
predictions = None
try:
    from synapse.ml.predict import MLFlowTransformer
    spark_scoring = spark.createDataFrame(spark_features)
    model_transformer = MLFlowTransformer(
        inputCols=list(params.feature_columns),
        outputCol="Churn_Prediction",
        modelName="freshmart-churn-model",
        modelVersion=1,
    )
    df_predictions = model_transformer.transform(spark_scoring)
    predictions = df_predictions.toPandas()
    print("Scored with MLFlowTransformer / PREDICT")
except Exception as exc:
    print(f"PREDICT unavailable ({exc}). Using local/registry fallback.")
    try:
        import mlflow
        pyfunc_model = mlflow.pyfunc.load_model("models:/freshmart-churn-model/1")
        preds = pyfunc_model.predict(model_matrix(spark_features, params))
        predictions = spark_features.copy()
        predictions["Churn_Prediction"] = preds
        print("Scored with mlflow.pyfunc")
    except Exception as inner:
        print(f"MLflow registry unavailable ({inner}). Retrain local champion for smoke test.")
        from sklearn.ensemble import RandomForestClassifier
        silver = load_table_or_csv("silver.customer_features", ".local/silver_customer_features.csv")
        if "MembershipTier_Bronze" not in silver.columns:
            raw = load_table_or_csv("bronze.customers", "freshmart_customers.csv")
            silver = transform_customers(raw, params, require_target=True)
        model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
        model.fit(model_matrix(silver, params), silver["Churn"].astype(int))
        predictions = spark_features.copy()
        predictions["Churn_Prediction"] = model.predict(model_matrix(spark_features, params))

print(predictions[["CustomerID", "RecencyDays", "MonetaryTotal", "ComplaintCount", "Churn_Prediction"]].head(10))


### ขั้นตอนที่ 3: เขียนตาราง Gold + ความเร่งด่วนแคมเปญ

**โค้ดนี้ทำอะไร**
- เพิ่ม `Action_Priority`: ค่าทำนาย `1` เป็น High (ส่ง voucher), ค่า `0` เป็น Normal
- เขียนตาราง Delta ชื่อ `gold.freshmart_predictions`

**สิ่งที่ควรเห็น:** 200 แถว และมีทั้งสองกลุ่มความเร่งด่วน  
(ชุดอ้างอิงท้องถิ่นประมาณ 36 High / 164 Normal — บน Fabric อาจขยับเล็กน้อย)

จบแล็บเมื่อพิมพ์ `Lab 4 verification passed`


In [ ]:
from datetime import datetime, timezone

gold = predictions.copy()
gold["Scored_Timestamp"] = datetime.now(timezone.utc).isoformat()
gold["Action_Priority"] = gold["Churn_Prediction"].map({
    1: "High - Send Retention Voucher",
    0: "Normal - Standard Engagement",
})
assert set(gold["Churn_Prediction"].unique()) <= {0, 1}
print(gold["Action_Priority"].value_counts())

try:
    (
        spark.createDataFrame(gold)
        .write.format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable("gold.freshmart_predictions")
    )
    print("Published gold.freshmart_predictions")
    spark.sql("""
        SELECT Action_Priority, COUNT(*) AS CustomerCount
        FROM gold.freshmart_predictions
        GROUP BY Action_Priority
    """).show()
except Exception as exc:
    out = Path("labs/data/.local/gold_freshmart_predictions.csv")
    out.parent.mkdir(parents=True, exist_ok=True)
    gold.to_csv(out, index=False)
    print(f"Spark write unavailable ({exc}). Wrote {out}")

assert len(gold) == 200
print("Lab 4 verification passed")
